In [1]:
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from env import (SingleLeafThreadEnv, ThreadingConfig)
from loss import TrajectoryBalanceLoss
from viz import draw_tree_edge_index, draw_local_tree_sequence
from utils import (
    get_max_actions,
    load_argweaver_sites_graph_inputs,
    graph_segments_to_physical_tskit,
    write_graph_segments_json,
)
from models import Policy


In [ ]:
SITES_PATH = "./validation_scripts/scripts/argweaver/argweaver_input/sim_l25kb_0.sites"
TREES_PATH = "./validation_scripts/trees/sim_l25kb_0.trees"
TIME_GRID_SIZE = 10

GENO, REFERENCE_GRAPH_SEGMENTS, LEAF_NAMES, ALL_LEAF_IDS, TIME_GRID = load_argweaver_sites_graph_inputs(
    SITES_PATH,
    TREES_PATH,
    time_grid_size=TIME_GRID_SIZE,
)


In [13]:
TIME_GRID

(0.0,
 9299.836184777578,
 18599.672369555155,
 27899.508554332733,
 37199.34473911031,
 46499.18092388789,
 55799.01710866547,
 65098.85329344304,
 74398.68947822062,
 83698.5256629982)

In [3]:
ALL_LEAF_IDS[-1]

7

In [7]:
## Env

N_EPISODES = 5
FOCAL_LEAF_ORDER = [ALL_LEAF_IDS[i % len(ALL_LEAF_IDS)] for i in range(N_EPISODES)]
PRINT_ACTIONS = False
SHOW_LOCAL_TREES = False
env_cfg = ThreadingConfig.from_argweaver_sites(
    SITES_PATH,
    TIME_GRID,
    mutation_rate=0.4,
    recomb_rate=0.35,
    reward_temperature=0.15,
    substitution_model="JC",
)
env = SingleLeafThreadEnv(env_cfg, ALL_LEAF_IDS, REFERENCE_GRAPH_SEGMENTS)


In [8]:
## Policy
policy_model = Policy(LEAF_NAMES, time_grid=TIME_GRID, device='cuda')

In [9]:
## N inner episodes with Trajectory Balance and terminal total reward
## One episode = one focal leaf threaded across all sites.


if not hasattr(policy_model, "log_Z"):
    raise RuntimeError("Policy is missing log_Z; rerun the Policy cell after reloading models.py.")
if "tb_loss" not in globals():
    tb_loss = TrajectoryBalanceLoss()


def _optimizer_matches_model(optimizer, model):
    model_params = {id(param) for param in model.parameters()}
    optimizer_params = {id(param) for group in optimizer.param_groups for param in group["params"]}
    return model_params == optimizer_params


if "optimizer" not in globals() or not _optimizer_matches_model(optimizer, policy_model):
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-3)


def _choice_key(choice):
    return choice.branch_child, choice.branch_signature, choice.time_idx, choice.is_root_branch


policy_model.train()
episode_summaries = []

for episode_idx in range(N_EPISODES):
    episode_focal_leaf = FOCAL_LEAF_ORDER[episode_idx]
    st = env.reset(focal_leaf=episode_focal_leaf)
    sum_log_P_F = None
    episode_actions = []
    total_local_log_reward = 0.0
    done = False
    step_idx = 0

    while not done:
        assert env._inner_env is not None and st.inner_state is not None
        inner_env = env._inner_env
        inner_state = st.inner_state
        site_idx = inner_state.site_index

        valid_acts = env.valid_actions(st)
        if not valid_acts:
            raise RuntimeError(f"No valid actions at episode {episode_idx}, step {step_idx}")

        # Keep action_choices aligned with valid_acts: row j in action_probs maps to valid_acts[j].
        action_choices = [inner_env.site_choices[site_idx][action_idx] for action_idx in valid_acts]

        site_graph = inner_env._site_tree_for_encoding(inner_state)
        action_logits, action_probs, edge_features, node_embeddings, leaf_feature = policy_model(
            site_graph,
            st.current_focal_leaf,
            action_choices,
        )

        focal_name = LEAF_NAMES[st.current_focal_leaf]
        probs_cpu = action_probs.detach().cpu()
        if PRINT_ACTIONS:
            print(f"\n=== episode {episode_idx + 1}/{N_EPISODES} | leaf {focal_name} | site {site_idx} ===")
            print("Action probabilities:")
            for row_idx, (action_idx, prob) in enumerate(zip(valid_acts, probs_cpu)):
                desc = env.describe_action(st, action_idx, leaf_names=LEAF_NAMES)
                print(f"  row {row_idx:02d} -> env action {action_idx:02d} | p={prob.item():.4f} | {desc}")

        dist = torch.distributions.Categorical(probs=action_probs)
        selected_row = dist.sample()
        selected_row_idx = int(selected_row.item())
        selected_action = valid_acts[selected_row_idx]
        selected_choice = action_choices[selected_row_idx]
        selected_log_prob = dist.log_prob(selected_row)
        sum_log_P_F = selected_log_prob if sum_log_P_F is None else sum_log_P_F + selected_log_prob
        if PRINT_ACTIONS:
            selected_desc = env.describe_action(st, selected_action, leaf_names=LEAF_NAMES)
            print(f"Selected: row {selected_row_idx} -> env action {selected_action} | {selected_desc}")

        st, local_log_reward, done = env.step(st, selected_action)

        total_local_log_reward += float(local_log_reward)

        if PRINT_ACTIONS:
            episode_actions.append(
                {
                    "step": step_idx,
                    "leaf": focal_name,
                    "site": site_idx,
                    "row": selected_row_idx,
                    "env_action": int(selected_action),
                    "prob": float(probs_cpu[selected_row_idx].item()),
                    "local_log_reward": float(local_log_reward),
                    "description": selected_desc,
                }
            )

        if SHOW_LOCAL_TREES:
            recomb_count = inner_state.recomb_count
            if inner_state.choices and _choice_key(selected_choice) != _choice_key(inner_state.choices[-1]):
                recomb_count += 1
            preview_inner_state = type(inner_state)(
                site_index=site_idx + 1,
                choices=inner_state.choices + (selected_choice,),
                recomb_count=recomb_count,
            )
            local_trees = inner_env.snapshot_state(preview_inner_state)
            if local_trees:
                draw_local_tree_sequence(
                    local_trees,
                    leaf_names=LEAF_NAMES,
                    time_grid=TIME_GRID,
                    use_time_as_y=True,
                )
                plt.show()

        step_idx += 1

    assert env._inner_env is not None and st.inner_state is not None
    if sum_log_P_F is None:
        raise RuntimeError("Episode ended before sampling any actions")
    total_log_reward = torch.tensor(
        env._inner_env.compute_log_reward(st.inner_state),
        dtype=sum_log_P_F.dtype,
        device=sum_log_P_F.device,
    )
    additive_log_reward = torch.tensor(total_local_log_reward, dtype=sum_log_P_F.dtype, device=sum_log_P_F.device)
    if not torch.allclose(total_log_reward, additive_log_reward, atol=1e-5):
        raise RuntimeError(
            "Terminal total reward does not match accumulated local rewards: "
            f"{total_log_reward.item():.4f} vs {additive_log_reward.item():.4f}"
        )

    loss = tb_loss(
        policy_model.log_Z,
        sum_log_P_F.reshape(1),
        total_log_reward.reshape(1),
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    summary = {
        "episode": episode_idx + 1,
        "focal_leaf": int(episode_focal_leaf),
        "steps": step_idx,
        "sum_log_P_F": float(sum_log_P_F.item()),
        "total_log_reward": float(total_log_reward.item()),
        "loss": float(loss.item()),
        "leaves_threaded": st.leaves_threaded,
        "actions": episode_actions,
        "graph_segments": st.current_graph_segments,
    }
    episode_summaries.append(summary)
    print(
        f"episode {episode_idx + 1:02d}/{N_EPISODES} | "
        f"leaf {LEAF_NAMES[episode_focal_leaf]} | steps {step_idx} | "
        f"sum_log_P_F {sum_log_P_F.item():.4f} | "
        f"total_log_reward {total_log_reward.item():.4f} | loss {loss.item():.4f}"
    )

print("\n=== training summary ===")
print(f"episodes: {len(episode_summaries)}")
print("focal leaf order: " + ", ".join(f"{LEAF_NAMES[lid]} ({lid})" for lid in FOCAL_LEAF_ORDER))
print(f"final log_Z: {policy_model.log_Z.item():.4f}")
print(f"final graph segments: {len(env.current_graph_segments)}")


episode 01/5 | leaf spl0_1 | steps 36 | sum_log_P_F -98.0331 | total_log_reward -8089.7583 | loss 63867672.0000
episode 02/5 | leaf spl0_2 | steps 36 | sum_log_P_F -99.3868 | total_log_reward -8406.9209 | loss 69015104.0000
episode 03/5 | leaf spl1_1 | steps 36 | sum_log_P_F -105.3503 | total_log_reward -8239.1074 | loss 66157976.0000
episode 04/5 | leaf spl1_2 | steps 36 | sum_log_P_F -111.2267 | total_log_reward -8592.1074 | loss 71925288.0000
episode 05/5 | leaf spl2_1 | steps 36 | sum_log_P_F -116.6833 | total_log_reward -7857.3638 | loss 59918068.0000

=== training summary ===
episodes: 5
focal leaf order: spl0_1 (0), spl0_2 (1), spl1_1 (2), spl1_2 (3), spl2_1 (4)
final log_Z: -0.0050
final graph segments: 36


In [10]:
## Save trained policy
from pathlib import Path

CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "policy.pt"

torch.save(
    {
        "state_dict": policy_model.state_dict(),
        "leaf_names": LEAF_NAMES,
        "time_grid": list(TIME_GRID),
        "log_Z": float(policy_model.log_Z.item()),
        "n_episodes_trained": N_EPISODES,
    },
    CHECKPOINT_PATH,
)
print(f"Saved policy to {CHECKPOINT_PATH}")

Saved policy to checkpoints/policy.pt


In [11]:
## Load checkpoint, sample diverse trajectories, and save physical-coordinate .trees outputs
from pathlib import Path
import json
from gflownet import TrajectorySampler
from utils import graph_segments_to_physical_tskit, write_graph_segments_json

loaded_policy = Policy(LEAF_NAMES, time_grid=TIME_GRID, device='cuda')
payload = torch.load(CHECKPOINT_PATH, map_location='cuda')
state_dict = payload.get("state_dict", payload) if isinstance(payload, dict) else payload
loaded_policy.load_state_dict(state_dict)
loaded_policy.eval()

sampler = TrajectorySampler(env, loaded_policy, reset_to_reference=True)

N_SAMPLES = 100
focal_leaf_for_demo = ALL_LEAF_IDS[0]
OUTPUT_DIR = Path("./validation_scripts/scripts/drag/l25kb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
physical_sequence_length = (
    float(env.config.genomic_region[1])
    if env.config.genomic_region is not None
    else float(env.config.sequence_length)
)

sampled_trajectories = []
trace_rows = []

print(f"Sampling {N_SAMPLES} trajectories for focal leaf {LEAF_NAMES[focal_leaf_for_demo]}:")
for i in range(N_SAMPLES):
    traj = sampler.sample(
        focal_leaf=focal_leaf_for_demo,
        temperature=1.0,
        greedy=False,
        reset_to_reference=True,
    )
    sampled_trajectories.append(traj)

    trees_path = OUTPUT_DIR / f"sample_{i:03d}.trees"
    segments_path = OUTPUT_DIR / f"sample_{i:03d}.segments.json"
    segments = traj.terminal_state.current_graph_segments
    graph_segments_to_physical_tskit(
        segments,
        num_samples=len(ALL_LEAF_IDS),
        sequence_length=physical_sequence_length,
        output_path=trees_path,
    )
    write_graph_segments_json(
        segments_path,
        segments,
        sequence_length=physical_sequence_length,
        num_samples=len(ALL_LEAF_IDS),
        coordinate_mode="physical",
    )

    trace_rows.append(
        {
            "sample_index": i,
            "focal_leaf": int(focal_leaf_for_demo),
            "focal_leaf_name": LEAF_NAMES[focal_leaf_for_demo],
            "log_reward": float(traj.log_reward.detach().cpu().item()),
            "sum_log_pf": float(traj.sum_log_pf.detach().cpu().item()),
            "recomb_count": int(traj.recomb_count),
            "length": int(traj.length),
            "checkpoint_path": str(CHECKPOINT_PATH),
            "sites_path": str(SITES_PATH),
            "trees_path": str(TREES_PATH),
            "sample_trees_path": str(trees_path),
            "segments_json_path": str(segments_path),
        }
    )

    actions_preview = traj.actions[:8]
    suffix = "..." if traj.length > 8 else ""
    print(
        f"  sample {i+1}: log_reward={traj.log_reward.item():.4f} | "
        f"sum_log_pf={traj.sum_log_pf.item():.4f} | "
        f"recombs={traj.recomb_count} | length={traj.length} | "
        f"saved={trees_path} | actions={actions_preview}{suffix}"
    )

trace_path = OUTPUT_DIR / "ours_trace.json"
with trace_path.open("w", encoding="utf-8") as handle:
    json.dump(trace_rows, handle, indent=2)
print(f"Saved {len(sampled_trajectories)} sampled .trees files and trace to {OUTPUT_DIR}")


Sampling 100 trajectories for focal leaf spl0_1:
  sample 1: log_reward=-8398.8271 | sum_log_pf=-98.0602 | recombs=33 | length=36 | saved=validation_scripts/scripts/drag/l25kb/sample_000.trees | actions=(3, 7, 9, 3, 10, 4, 0, 1)...
  sample 2: log_reward=-8415.8682 | sum_log_pf=-97.8911 | recombs=33 | length=36 | saved=validation_scripts/scripts/drag/l25kb/sample_001.trees | actions=(5, 6, 1, 11, 9, 4, 14, 3)...
  sample 3: log_reward=-8263.9414 | sum_log_pf=-98.0970 | recombs=34 | length=36 | saved=validation_scripts/scripts/drag/l25kb/sample_002.trees | actions=(0, 3, 2, 4, 4, 8, 6, 11)...
  sample 4: log_reward=-7865.5786 | sum_log_pf=-98.0517 | recombs=35 | length=36 | saved=validation_scripts/scripts/drag/l25kb/sample_003.trees | actions=(3, 8, 4, 3, 11, 7, 2, 5)...
  sample 5: log_reward=-7871.5474 | sum_log_pf=-98.0622 | recombs=33 | length=36 | saved=validation_scripts/scripts/drag/l25kb/sample_004.trees | actions=(2, 1, 4, 0, 4, 4, 11, 14)...
  sample 6: log_reward=-8221.8838 

KeyboardInterrupt: 